# Predicción horaria con VS-GRU

VS-GRU procesa cada variable temporal por separado, con un estado recurrente por variable. La celda imputa los faltantes por decaimiento hacia la media de entrenamiento y usa la proporción de faltantes dentro de las horas válidas de cada ventana. El clasificador combina los estados finales con el contexto del paciente.

Antes de empezar, ejecuta `uv sync --extra model --group dev` en la raíz del proyecto, selecciona el kernel de `.venv` y comprueba que `cleaning.ipynb` haya exportado las particiones.

In [ ]:
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
from tensorflow import keras

from appr_project.data import (
    PatientHours,
    Standardization,
    WindowBatches,
    load_partition,
    split_patients,
)
from appr_project.evaluation import predict_hours, summarize_metrics
from appr_project.training import Hyperparameters, train_experiment

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
DATA_DIR = ROOT / "data" / "processed"

## 1. Configuración del piloto

El piloto usa hasta 2.000 pacientes de A y una sola partición de validación. Los dos ensayos comparten partición e hiperparámetros y comparan una salida directa con una capa oculta de 32 neuronas y ReLU. El modelo se selecciona por precisión promedio (AP) en validación.

Para medir el aporte de B con la arquitectura fija, cambia `source` a `"A+B"`, pon `trials=1` y copia el valor ganador de `use_hidden_dense` a `base`. Usa la misma configuración de muestreo y partición que en los notebooks semánticos para comparar arquitecturas dentro de cada origen.

In [ ]:
settings = {
    "architecture": "vs_gru",
    "source": "A",
    "sample_patients": 2000,
    "validation_fraction": 0.20,
    "folds": None,
    "search": "random",
    "trials": 2,
    "epochs": 30,
    "patience": 4,
    "seed": 2026,
}

base = Hyperparameters(
    window=12,
    learning_rate=0.001,
    batch_size=128,
    use_hidden_dense=False,
)

## 2. Inspeccionar la partición y un batch

Se muestran los pacientes, sus horas y cuántos tienen alguna etiqueta positiva. En CV, la vista previa corresponde al primer fold. La muestra y la partición se reproducen al entrenar con la misma semilla.

Cada entrada temporal tiene dimensiones `[batch, horas, variables, 4]`. Los cuatro canales son el valor estandarizado, la máscara de medición, el tiempo desde la medición previa y su valor. La máscara `valid_hours` marca las horas reales de la ventana.


In [ ]:
frame = load_partition(DATA_DIR, "train", settings["source"])

train_ids, val_ids = next(
    split_patients(
        frame,
        validation_fraction=settings["validation_fraction"],
        folds=settings["folds"],
        seed=settings["seed"],
        sample_patients=settings["sample_patients"],
    )
)

train_preview = frame.loc[frame.patient_id.isin(train_ids)]
val_preview = frame.loc[frame.patient_id.isin(val_ids)]

summary = []
for name, partition in (("train", train_preview), ("validation", val_preview)):
    patient_labels = partition.groupby("patient_id").SepsisLabel.max()
    summary.append(
        {
            "partición": name,
            "pacientes": len(patient_labels),
            "pacientes positivos": int(patient_labels.sum()),
            "horas": len(partition),
        }
    )
display(pd.DataFrame(summary).set_index("partición"))

In [ ]:
scaler = Standardization.fit(train_preview)
first_patient = train_preview.loc[train_preview.patient_id.eq(train_ids[0])]
hours = PatientHours(first_patient, scaler)
batches = WindowBatches(hours, base.window, base.batch_size)
inputs, labels = batches[0]

print("Formas de entrada:", {name: array.shape for name, array in inputs.items()})
print("Forma de las etiquetas:", labels.shape)
print("Paciente:", hours.metadata["patient_id"].iloc[0])
print(
    "Horas válidas de la primera ventana:",
    inputs["valid_hours"][0].astype(int).tolist(),
)
print("Etiqueta de la primera ventana:", int(labels[0, 0]))

print("Variables temporales:", scaler.temporal)

## 3. Entrenar y revisar los resultados

Cada ejecución guarda el modelo, la configuración, las particiones, las métricas y los tiempos en `runs/`. La columna `use_hidden_dense` identifica la variante de cada ensayo.

Para comparar contra `modeling_semantic.ipynb` y `modeling_semantic_3.ipynb`, comprueba que coincidan las asignaciones de pacientes en `splits.csv` y usa el mismo origen de prueba.

In [ ]:
run_dir = (
    ROOT
    / "runs"
    / f"notebook-vs-gru-{settings['source']}-{datetime.now():%Y%m%d-%H%M%S-%f}"
)
validation_metrics = train_experiment(frame, run_dir, base=base, **settings)

print(f"Resultados: {run_dir}")
display(validation_metrics.round(4))

In [ ]:
trial_results = pd.read_csv(run_dir / "trials.csv")
display(
    trial_results[
        ["trial", "use_hidden_dense", "mean_validation_ap", "std_validation_ap"]
    ].round(4)
)
config = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
print("Capa oculta seleccionada:", config["hyperparameters"]["use_hidden_dense"])

fold_metrics = pd.read_csv(run_dir / "fold_metrics.csv")
timing_columns = [
    "trial",
    "fold",
    "train_windows",
    "validation_windows",
    "preparation_seconds",
    "epochs_run",
    "seconds_per_epoch",
]
display(fold_metrics.loc[fold_metrics.source.eq("all"), timing_columns].round(2))


run_timing = json.loads((run_dir / "run_timing.json").read_text(encoding="utf-8"))
print(f"Tiempo total de ejecución: {run_timing['total_seconds']:.2f} s")

## 4. Evaluación final opcional

Activa esta celda cuando hayas elegido la configuración definitiva. La utilidad y la proporción de horas con alerta en test se calculan con el umbral guardado en la corrida. Los resultados se muestran y se guardan por origen y de forma agregada. Usa `positive_hour_fraction` como referencia para el AP.


In [ ]:
evaluate_test = False
test_source = "A+B"  # También admite "A" o "B".

if evaluate_test:
    config = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
    saved_model = keras.models.load_model(run_dir / "best.keras", compile=False)
    saved_scaler = Standardization(**config["standardization"])
    test = load_partition(DATA_DIR, "test", test_source)
    test_batches = WindowBatches(
        PatientHours(test, saved_scaler),
        config["hyperparameters"]["window"],
        config["hyperparameters"].get("batch_size", config.get("batch_size")),
    )
    test_predictions = predict_hours(saved_model, test_batches)
    test_metrics = summarize_metrics(test_predictions, config["threshold"])
    test_predictions.to_csv(
        run_dir / f"test_{test_source}_predictions.csv", index=False
    )
    test_metrics.to_csv(run_dir / f"test_{test_source}_metrics.csv", index=False)
    display(test_metrics.round(4))